# CH-SW Fit

Long-range CH-driven solar-wind propagation notebook for fitting workflows using sparse-phi propagation.

This notebook propagates from `2010-01-01` through the end of `2024` with `phi_targets = [0.0]`.

It uses the standard ballistic solver on a sparse phi axis, so the `phi=0` slice matches the full propagation at the same cadence and model settings.

For long-range fitting throughput, `superresolution_enabled_override` is set to `False` by default here. Set it to `None` to match `Config/SW/Ballistic.json` exactly.

In [45]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/helio_n_matplotlib")

cwd = Path.cwd().resolve()
project_root = None
for candidate in (cwd, *cwd.parents, Path("/home/smdc/helio-n")):
    if (candidate / "Library").exists() and (candidate / "Config").exists():
        project_root = candidate
        break
assert project_root is not None, "Could not locate the helio-n project root for notebook imports."
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from Library.SW.Ballistic import postprocess_max_field, propagate_phi_targets
from Library.SW.Config import (
    load_ballistic_spec,
    load_empirical_spec,
    load_sw_runtime_spec,
    resolve_time_controls,
)
from Library.SW.Coords import compute_rotation_state
from Library.SW.Inputs import (
    build_forecast_earth_frame,
    build_model_input_series,
    load_ace_at_earth,
    load_sw_input_frame,
    load_sw_input_from_sql
)
from Library.SW.Visualization import (
    build_earth_comparison_frame,
    plot_full_range_earth_comparison,
    plot_time_radius_slice,
)


In [ ]:
# handoff_dt = pd.Timestamp("2019-01-01")
# df_sql_handoff = load_sw_input_from_sql("2018-06-01", "2019-06-01")
# df_sql_handoff["forecast_dt"] = pd.to_datetime(df_sql_handoff["forecast_dt"])
# df_sql_handoff = df_sql_handoff.sort_values("forecast_dt")

# handoff_daily = (
#     df_sql_handoff[["forecast_dt", "ch_relative_area", "forecast_sw_speed"]]
#     .groupby("forecast_dt", as_index=True)
#     .mean()
#     .resample("1D")
#     .mean()
# )
# handoff_smooth = handoff_daily.rolling("14D", center=True, min_periods=1).mean()

# fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True, constrained_layout=True)

# axes[0].plot(
#     handoff_daily.index,
#     handoff_daily["ch_relative_area"],
#     color="0.75",
#     linewidth=1.0,
#     label="daily mean",
# )
# axes[0].plot(
#     handoff_smooth.index,
#     handoff_smooth["ch_relative_area"],
#     color="tab:blue",
#     linewidth=2.0,
#     label="14-day rolling mean",
# )
# axes[0].axvline(handoff_dt, color="crimson", linestyle="--", linewidth=2)
# axes[0].axvspan(handoff_dt - pd.Timedelta(days=7), handoff_dt + pd.Timedelta(days=7), color="crimson", alpha=0.08)
# axes[0].set_ylabel("CH relative correct sphere area")
# axes[0].legend(loc="upper right")
# axes[0].set_title("SQL handoff diagnostic around 2019-01-01")
# axes[0].grid(alpha=0.25)

# axes[1].plot(
#     handoff_daily.index,
#     handoff_daily["forecast_sw_speed"],
#     color="0.75",
#     linewidth=1.0,
#     label="daily mean",
# )
# axes[1].plot(
#     handoff_smooth.index,
#     handoff_smooth["forecast_sw_speed"],
#     color="tab:green",
#     linewidth=2.0,
#     label="14-day rolling mean",
# )
# axes[1].axvline(handoff_dt, color="crimson", linestyle="--", linewidth=2, label="2019-01-01")
# axes[1].axvspan(handoff_dt - pd.Timedelta(days=7), handoff_dt + pd.Timedelta(days=7), color="crimson", alpha=0.08)
# axes[1].set_ylabel("Forecast SW speed [km/s]")
# axes[1].set_xlabel("Forecast date")
# axes[1].legend(loc="upper right")
# axes[1].grid(alpha=0.25)

# plt.show()

# handoff_daily.loc[handoff_dt - pd.Timedelta(days=14): handoff_dt + pd.Timedelta(days=14)]


In [37]:
start_dt = pd.Timestamp("2018-01-01")
end_dt = pd.Timestamp("2026-03-31")
input_source = "sql"
# input_parquet_path = "Data/CH Area.parquet"
phi_targets = [0.0]
superresolution_enabled_override = True
show_progress = True

print("start_dt:", start_dt)
print("end_dt:", end_dt)
print("phi_targets:", phi_targets)
print("superresolution_enabled_override:", superresolution_enabled_override)


start_dt: 2018-01-01 00:00:00
end_dt: 2026-03-31 00:00:00
phi_targets: [0.0]
superresolution_enabled_override: True


In [44]:
empirical = load_empirical_spec()
empirical

EmpiricalSpec(json_path=PosixPath('/home/smdc/helio-n/Config/SW/Empirical.json'), slow_sw_speed=300.0, v_min=300.0, a=180.0, alpha=0.6)

In [ ]:
ballistic = load_ballistic_spec()
runtime = load_sw_runtime_spec()
time_controls = resolve_time_controls(
    ballistic,
    superresolution_enabled_override=superresolution_enabled_override,
)

df_sdo_sw = load_sw_input_frame(
    start_dt=start_dt,
    end_dt=end_dt,
    source=input_source,
    # input_parquet_path=input_parquet_path,
)


In [39]:
prepared = build_model_input_series(
    sdo_input_df=df_sdo_sw,
    empirical=empirical,
    time_controls=time_controls,
    simulation_pad_days=ballistic.simulation_pad_days,
    input_chunk_rows=runtime.input_chunk_rows,
    show_progress=show_progress,
)
rotation = compute_rotation_state(
    cr_days=ballistic.cr_days,
    phi_step_minutes=ballistic.phi_step_minutes,
)
df_v_run = prepared["df_v"].loc[
    (prepared["df_v"].index >= prepared["sim_start"])
    & (prepared["df_v"].index <= prepared["sim_end"])
].copy()

print("Empirical.json:", empirical.json_path)
print("Ballistic.json:", ballistic.json_path)
print("Machine.json[%r].sw:" % runtime.machine_name, runtime.machine_json_path)
print("input_chunk_rows:", runtime.input_chunk_rows)
print("time_freq:", time_controls.time_freq)
print("Seeds in run:", len(df_v_run))
print("Simulation range:", prepared["sim_start"], "->", prepared["sim_end"])


empirical v(area):   0%|          | 0/1 [00:00<?, ?rows/s]

Empirical.json: /home/smdc/helio-n/Config/SW/Empirical.json
Ballistic.json: /home/smdc/helio-n/Config/SW/Ballistic.json
Machine.json['miracle'].sw: /home/smdc/helio-n/Config/Machine.json
input_chunk_rows: 200000
time_freq: 5min
Seeds in run: 866185
Simulation range: 2018-01-01 00:00:00 -> 2026-04-03 14:00:00


In [40]:
phi_run = propagate_phi_targets(
    df_v_run=df_v_run,
    sim_start=prepared["sim_start"],
    sim_end=prepared["sim_end"],
    time_freq=time_controls.time_freq,
    rotation_state=rotation,
    r0=ballistic.r0,
    r_max=ballistic.r_max,
    prop_stats_mode=ballistic.prop_stats_mode,
    dense_memory_budget_gb=runtime.dense_memory_budget_gb,
    memory_guard_enabled=ballistic.memory_guard_enabled,
    horizon_hours=ballistic.horizon_hours,
    time_step_hours=time_controls.time_step_hours,
    field_half_width_h=ballistic.field_half_width_h,
    r_solar_km=ballistic.r_solar_km,
    use_swept_cell=ballistic.use_swept_cell,
    use_cr_reset=ballistic.use_cr_reset,
    max_seed_batch=runtime.max_seed_batch,
    phi_targets=phi_targets,
    show_progress=show_progress,
)

print("Propagation runtime (s):", phi_run.stats.prop_seconds)
print("Seeds processed:", phi_run.stats.seeds_processed)
print("Filled cells:", phi_run.stats.filled, "/", phi_run.stats.total)
print("Grid shape:", phi_run.accumulators.V_accum_max.shape)
print("Resolved phi axis:", phi_run.grid.phi_axis)


2D propagate:   0%|          | 0/423 [00:00<?, ?batch/s]

Propagation runtime (s): 58.389864487573504
Seeds processed: 866185
Filled cells: 169352286 / 170167396
Grid shape: (868201, 1, 196)
Resolved phi axis: [0.]


In [42]:
post = postprocess_max_field(
    V_accum_max=phi_run.accumulators.V_accum_max,
    slow_sw_speed=empirical.slow_sw_speed,
    post_chunk_t=runtime.post_chunk_t,
    show_progress=show_progress,
)

time_axis = phi_run.grid.time_axis
phi_axis = phi_run.grid.phi_axis
r_axis = phi_run.grid.r_axis
V_grid = post.V_grid
post_fields_raw = post.post_fields_raw
slow_sw_pred_mask = post.slow_sw_pred_mask

earth_r_idx = int(np.argmin(np.abs(r_axis - ballistic.earth_r_target)))
propagated_earth = pd.Series(
    post_fields_raw["max"][:, 0, earth_r_idx],
    index=time_axis,
    name="v_model_earth_phi0",
)

print("Postprocessed grid shape:", V_grid.shape)
print("Earth target r idx:", earth_r_idx)
propagated_earth.to_frame()


Post-processing:   0%|          | 0/1696 [00:00<?, ?chunk/s]

Postprocessed grid shape: (868201, 1, 196)
Earth target r idx: 195


,v_model_earth_phi0
2018-01-01 00:00:00,NaN
2018-01-01 00:05:00,NaN
2018-01-01 00:10:00,NaN
2018-01-01 00:15:00,NaN
2018-01-01 00:20:00,NaN
...,...
2026-04-03 13:40:00,NaN
2026-04-03 13:45:00,NaN
2026-04-03 13:50:00,NaN
2026-04-03 13:55:00,NaN


In [ ]:
# Reorganize this notebook so that the entire propagation is wrapped in a single function with the components to EmpiricalSpec slow_sw_speed=300.0, v_min=300.0, a=180.0, alpha=0.6 as its arguments. Obviously DB access outside of the fumction. It's prep for applying something like FindFit to it